In [ ]:
import json
import pandas as pd
import os

In [3]:
# Load the raw data
with open(f'data/raw/t1d_phase3_trials_2026-10-24.json') as f:
    trials = json.load(f)

In [4]:
# define a function to flatten the trial data
def flatten_trial(trial):
    ps = trial['protocolSection']
    row = {
        'nct_id': ps['identificationModule']['nctId'],
        'status': ps['statusModule']['overallStatus'],
        'why_stopped': ps['statusModule'].get('whyStopped', None),
        'sponsor_class': ps['sponsorCollaboratorsModule']['leadSponsor']['class'],
        'lead_sponsor_name': ps['sponsorCollaboratorsModule']['leadSponsor']['name'],
        'start_date': ps['statusModule'].get('startDateStruct',{}).get('date', ''),
        'completion_date': ps['statusModule'].get('completionDateStruct',{}).get('date', ''),
        'last_update': ps['statusModule'].get('lastUpdateSubmitDate', ''),
        'enrollment': ps['designModule'].get('enrollmentInfo', {}).get('count', ''),
        'phases': '|'.join(ps['designModule'].get('phases','')),
        'conditions': '|'.join(ps['conditionsModule']['conditions']),
        'has_results': trial.get('hasResults', None),
        'brief_summary': ps.get('descriptionModule', {}).get('briefSummary', ''),
        'detailed_description': ps.get('descriptionModule', {}).get('detailedDescription', '')
    }
    return row

In [5]:
# iterate through the trials and flatten each one save them in a list, then create a DataFrame
rows = []

for trial in trials:
    row = flatten_trial(trial)
    rows.append(row)
df = pd.DataFrame(rows)
df

,nct_id,status,why_stopped,sponsor_class,lead_sponsor_name,start_date,completion_date,last_update,enrollment,phases,conditions,has_results,brief_summary,detailed_description
0,NCT02546401,COMPLETED,NaN,OTHER,"University Hospital, Brest",2015-09,2016-03,2025-11-26,22,PHASE3,Diabetes Type 1,False,The current study focuses on terms of use of I...,The functional insulin therapy is a method for...
1,NCT04879511,COMPLETED,NaN,OTHER,Ain Shams University,2018-02-10,2021-01-29,2021-05-06,80,PHASE2|PHASE3,Effect of Change in T1DM Glycemic Control,False,Cardiovascular disease is the leading cause of...,Type 1 diabetes mellitus (T1DM) is an autoimmu...
2,NCT03377699,COMPLETED,NaN,INDUSTRY,Novo Nordisk A/S,2017-11-22,2020-12-17,2023-01-19,225,PHASE3,"Diabetes|Diabetes Mellitus, Type 1",True,The investigators are doing this study to see ...,
3,NCT01959334,COMPLETED,NaN,INDUSTRY,Eli Lilly and Company,2013-09,2013-12,2019-08-14,75,PHASE3,Drug-specific Antibodies|Diabetes Mellitus,True,This study provides information on immunogenic...,"This study is a single center, randomized, lab..."
4,NCT03302234,COMPLETED,NaN,INDUSTRY,Merck Sharp & Dohme LLC,2017-12-14,2022-09-07,2023-09-25,568,PHASE3,"Carcinoma, Non-Small-Cell Lung",True,The purpose of this study is to determine the ...,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
414,NCT01513473,COMPLETED,NaN,INDUSTRY,Novo Nordisk A/S,2012-01-16,2013-07-30,2019-06-03,350,PHASE3,"Diabetes|Diabetes Mellitus, Type 1",True,"This trial is conducted in Africa, Asia, Europ...",
415,NCT04011020,RECRUITING,NaN,INDUSTRY,Throne Biotechnologies Inc.,2022-09-20,2028-12-20,2026-09-29,80,PHASE3,Type 1 Diabetes,False,Type 1 diabetes (T1D) is a T cell-mediated aut...,The SCE device is made of a hydrophobic materi...
416,NCT03338023,COMPLETED,NaN,INDUSTRY,Eli Lilly and Company,2018-03-23,2020-03-05,2021-03-03,272,PHASE3,Type 1 Diabetes,True,The purpose of this study is to compare long-a...,
417,NCT00853944,TERMINATED,Sponsor withdrew funding du to lack of enrollm...,OTHER,University of British Columbia,2009-07,2013-11-01,2017-10-27,12,PHASE3,Islet Transplantation,False,Islet transplantation requires a large number ...,Hypothesis: Sitagliptin taken for 12 months af...


In [7]:
# Filter out any conditions that do not relate to T1D and save to a new dataframe
t1_pattern = r'Type 1|\bType I\b|T1D|Type1|Insulin Resi'
t2_pattern = r'Type 2|\bType II\b|T2D|Type2'

mentions_t1 = df['conditions'].str.contains(t1_pattern, case=False, na=False)
mentions_t2 = df['conditions'].str.contains(t2_pattern, case=False, na=False)
mentions_diabetes = df['conditions'].str.contains(r'diabet', case=False, na=False)

keep = (mentions_t1 | mentions_diabetes) & ~(mentions_t2 & ~mentions_t1)
df_t1d = df[keep]

# Trials excluded by manual review (see README Methods)
excluded_ids = [
    'NCT00521820',  # T2D population per brief summary; generic "Diabetes Mellitus" condition label
]

df_t1d = df_t1d[~df_t1d['nct_id'].isin(excluded_ids)]
print(len(df_t1d))  # should be one less than before

355


In [8]:
# Save the flattened data to a CSV file
os.makedirs('data/processed', exist_ok=True)
df_t1d.to_csv(f'data/processed/t1d_phase3_trials_2026-10-24.csv', index=False)